---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [21]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: c:\Users\Lenovo\OneDrive\Bureau\Ingineria_AI
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [22]:
import pandas as pd
import random

corpus = pd.read_json(
    r"C:\Users\Lenovo\OneDrive\Bureau\Ingineria_AI\echochamber-project-team-1\data\cleaned\corpus_youtube_sample.jsonl",
    lines=True
)
print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[declicro] Mă nebunilor, Aproape toți ungurii pe care îi știu o să voteze cu Orban și ăia c
[PressOneRomania] A primit drept de practica de la Colegiul medicilor in 2008. A absovit in 1988, 
[RecorderRomania] Mereu la înălțimea așteptărilor noastre,sunteți voi Recorder!Felicitări vouă!Atâ


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [23]:
# modifica dupa preferinte

AXA_1 = "INSTITUTIONAL"
AXA_2 = "EPISTEMIC"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [30]:
AXA_1_DEFINITION = """
instituțional măsoară modul în care comentariul evaluează instituțiile (stat, justiție, alegeri, partide).

-2: instituțiile sunt descrise ca corupte, capturate sau dictatoriale
-1: critică instituțională secundară
0: fără evaluare instituțională
+1: apărare a instituțiilor/procedurilor, secundară
+2: respectarea legii, constituției și procedurilor este centrală

Indicatori negativi: „sistem corupt”, „mafie”, „dictatură”, „alegeri furate”.
Indicatori pozitivi: „lege”, „constituție”, „reguli”, „nimeni nu e mai presus de lege”.
"""
AXA_2_DEFINITION = """
epistemic măsoară modul în care comentariul explică evenimentele politice.

-2: explică politica prin conspirații, manipulare sau forțe ascunse
-1: sugerează manipulare/conspirație, dar nu central
0: fără explicație cauzală relevantă
+1: cere dovezi sau verificare, secundar
+2: dovezile și verificarea sunt centrale

Indicatori negativi: „regizat”, „din umbră”, „manipulat”.
Indicatori pozitivi: „unde sunt dovezile?”, „verificați”, „pe baza probelor”.
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [25]:
MINI_PROMPT = f"""
Ești un analist de discurs politic.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2
{AXA_2} = 0 / 1 / 2
DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}
REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
6. Nu atribui direct o bulă discursivă.
7. Returnează doar JSON valid.
FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""
print(MINI_PROMPT)


Ești un analist de discurs politic.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. INSTITUTIONAL
2. EPISTEMIC
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
INSTITUTIONAL = 0 / 1 / 2
EPISTEMIC = 0 / 1 / 2
DEFINIȚII:

instituțional măsoarămodul în care comentariul evaluează instituțiile (stat, justiție, alegeri, partide).

-2: instituțiile sunt descrise ca corupte, capturate sau dictatoriale
-1: critică instituțională secundară
0: fără evaluare instituțională
+1: apărare a instituțiilor/procedurilor, secundară
+2: respectarea legii, constituției și procedurilor este centrală

Indicatori negativi: „sistem corupt”, „mafie”, „dictatură”, „alegeri furate”.
Indicatori pozitivi: „lege”, „constituție”, „reguli”, „nimeni nu e mai presus de lege”.


epistemic măsoară modul în care comentariul explică evenimentel

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [31]:
TESTS = corpus.sample(5)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
296,yt_N5paaKW5FKc_Ugyb51L5Kdgbug2cbD54AaABAg,PressOneRomania,Molia. Metamorfoza „doctorului de suflete” Cri...,Ideea ar fi ca violul are o definitie clara. N...
136,yt__8uDDh_S4zk_UgxhizYCZ6-r-NO4xat4AaABAg,happyfishteleviziune,CENZURA PE INTERNET - Un rau necesar? | Aproxi...,"1:01:06 Care e problema?Orice cetățean,cu oric..."
295,yt_GwJr4VT7uMU_Ugz1hG77KABPvvHsTHB4AaABAg,USR-2024,Adevărul despre moțiunea împotriva Dianei Buzo...,Bravo doamna! De oameni ca dv are nevoie Roman...
418,yt_QUlbNQEw-So_UgzRbWBffx_koLxNtjp4AaABAg,NicusorDanRO,🟢 LIVE Discursul susținut în cadrul recepției ...,FELICITĂRI Dle PREȘEDINTE DAN si LA MULȚI ANI ...
383,yt_re0gkFt114A_UgwAcMlgCdx0sDUKzGF4AaABAg,turcescu111,Ne bagă iar în lock-down!,Buna Robert. Te rog frumos sa l întrebi fronta...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [32]:
USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: gemini-2.5-flash-lite


In [33]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [34]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
Ideea ar fi ca violul are o definitie clara. Nu stiu exact care e, dar am impresia ca presupune un act sexual neconsfintit. Si pentru atestarea violului stiu ca e nevoie de prelevare de mostre biologice. Leziuni la nivelul vaginului, etc... Si daca nu are din alea... Si alea trebuiesc prelevate mna... In termen de cateva ore sa zicem, ca dupa o saptamana nu se mai vede nimic. Dupa aia ar fi tentativa de viol. Unde iarasi cred ca trebuiesc probe biologice. Si apoi hartuirea sexuala. La hartuire sexuala... Cazul ala cu Louis C.K.. Dar acolo e mega complicat. Pentru ca apar problemele deontologice. Care nu sunt cum crede lumea. Adica lumea crede ca e simplu. Psihoterapeut, n-are voie. Pa! Discutabil. Pentru ca daca nu avea contract sa zicem... Si prin facultati apare ideea de a te antrena ca psihoterapeut fara bani. Un prieten, din astea. Ceea ce... Adica daca profesezi ca psihoterapeut sau faci facultate, probabil vorbesti despre asta. Si daca iesi la un date ca atare si zici

In [ ]:
## Pasul 6 — Interpretare scurtă
Completează în notebook, în 3–5 rânduri:
- Ce două axe ai ales?
- De ce le-ai ales?
- Modelul a returnat JSON corect?
- Care a fost cea mai mare problemă?
- Ce ai schimba în prompt?

## Pasul 6 + INTERPRETARE
-Am ales axele din curs: instututional si epistemic
-Am ales aceste axe, deoarece ele surprind doua dimensiuni centrale ale discursurlui politic online.
-Modelul a returnat json corect
-Indentificare gresita a target-ului si internpretare excesiva a comentariilor. Spre exemplu in ultimul comentariu unde a indentificat tinta ca "autoritățile sanitare" care nu apare in text.
-Poate as schimba in promt ce inseamna target si ca acesta sa fie mentionat explicit in comentariu.

